# audit_trail_services_3 — Unit / Invariant Tests

Self-contained validation for the silver output of `audit_trail_services_3_bronze_to_silver_v3.ipynb`. **Just run these cells top to bottom** — they don't depend on the cleaning notebook: re-declare constants -> read `processed/audit_trail_services_3_v2/` + one-pass metric aggregation -> define all tests -> run.

Read with `recursiveFileLookup=true` (V2's `date=X/` sub-dirs otherwise trigger partition-discovery errors).

In [ ]:
%%pyspark project.spark.compatibility
# ============================================================
# Setup for validation -- self-contained, does not depend on any earlier cell
# ============================================================
from pyspark.sql import SparkSession, functions as F
from pyspark.sql import Row
from pyspark.sql.types import StringType, StructType, StructField

spark = SparkSession.builder.appName("audit_trail_services_3_tests").getOrCreate()

# ---- Path: read from processed/ (already deduped) ----
# must match the cleaning notebook's OUT_BASE (versioned path)
SILVER_PROCESSED_PATH = "s3://enstream-lake-silver-dev/processed/audit_trail_services_3_v2/"

# ---- Constant metadata ----
SOURCE_NAME    = "MYSQL"
SCHEMA_VERSION = 1

# ---- Test-data exclusion lists (for the leakage check) ----
TEST_PARTNER_IDS = [
    "DEV_CAMARA_CLEARX", "dev_camara", "ens_batch_test", "enstream",
    "flashProd", "flashprod", "oleg-prod-jose",
]
TEST_SERVICE_PROVIDER_IDS = [
    "DEV_CAMARA_CLEARX", "def_integrity_sp", "ens_batch_test", "flash21",
    "flashprod", "oleg-prodsp-jose", "testServiceProvider",
]

# ---- Target schema (spec B convention, 23 columns) ----
FINAL_COLS = [
    "record_id", "operation", "mno", "msisdn", "phone_number_AT_hash", "encrypted_msisdn",
    "partner_id", "partner_name",
    "service_provider_id", "service_provider_name",
    "response_code", "source_ip",
    "api_timestamp", "date", "timestamp_ms",
    "brand", "industry", "api_type",
    "processing_time_ms",
    "correlation_id", "source_name", "source_event_id", "schema_version",
]

# ---- pad_id_to_10 (same rule as cleaning; used by the fixture tests) ----
def pad_id_to_10(col_name):
    return (
        F.when(F.col(col_name).rlike(r"^\d+$"),
               F.lpad(F.col(col_name), 10, "0"))
         .otherwise(F.col(col_name))
    )

print("Validation setup done.")


In [ ]:
%%pyspark project.spark.compatibility
# Load silver (processed) + one-pass aggregation of all metrics
df_silver = (
    spark.read
         .option("recursiveFileLookup", "true")
         .parquet(SILVER_PROCESSED_PATH)
         .cache()
)
_total = df_silver.count()
print(f"Loaded {_total:,} rows for testing")

_agg = df_silver.select(
    # ----- null counts (per column) -----
    *[F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}") for c in df_silver.columns],

    # ----- format violations -----
    F.sum((F.col("phone_number_AT_hash").isNotNull() &
           (~F.col("phone_number_AT_hash").rlike(r"^[a-f0-9]{64}$"))).cast("int")).alias("bad_hash_format"),
    F.sum((F.col("source_ip").isNotNull() &
           (~F.col("source_ip").rlike(r"^\d{1,3}(\.\d{1,3}){3}$"))).cast("int")).alias("bad_ipv4_format"),
    F.sum((F.col("encrypted_msisdn").isNotNull() &
           (~F.col("encrypted_msisdn").rlike(r"^[A-Za-z0-9+/=]+$"))).cast("int")).alias("bad_b64_format"),
    F.sum((F.col("record_id").isNotNull() &
           (~F.col("record_id").rlike(r"^[a-f0-9]{64}$"))).cast("int")).alias("bad_record_id_format"),

    # ----- padding violations (digit-only IDs must be exactly 10 chars) -----
    F.sum((F.col("partner_id").rlike(r"^\d+$") &
           (F.length(F.col("partner_id")) != 10)).cast("int")).alias("partner_id_digit_not_10"),
    F.sum((F.col("service_provider_id").rlike(r"^\d+$") &
           (F.length(F.col("service_provider_id")) != 10)).cast("int")).alias("sp_id_digit_not_10"),

    # ----- constants -----
    F.sum((F.col("source_name") != F.lit(SOURCE_NAME)).cast("int")).alias("bad_source_name"),
    F.sum((F.col("schema_version") != F.lit(SCHEMA_VERSION)).cast("int")).alias("bad_schema_version"),
    F.sum((F.col("correlation_id").isNotNull()).cast("int")).alias("correlation_id_non_null"),
    F.sum((F.col("correlation_id").isNotNull() &
           (~F.col("correlation_id").rlike(
               r"^gen-[a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12}$"))
          ).cast("int")).alias("bad_correlation_id_format"),

    # ----- varchar length constraints (from the source schema) -----
    F.sum((F.col("partner_id").isNotNull() & (F.length("partner_id") > 50)).cast("int")).alias("partner_id_too_long"),
    F.sum((F.col("service_provider_id").isNotNull() & (F.length("service_provider_id") > 50)).cast("int")).alias("sp_id_too_long"),
    F.sum((F.col("mno").isNotNull() & (F.length("mno") > 255)).cast("int")).alias("mno_too_long"),
    F.sum((F.col("msisdn").isNotNull() & (F.length("msisdn") > 20)).cast("int")).alias("msisdn_too_long"),
    F.sum((F.col("phone_number_AT_hash").isNotNull() & (F.length("phone_number_AT_hash") > 70)).cast("int")).alias("hashed_msisdn_too_long"),
    F.sum((F.col("encrypted_msisdn").isNotNull() & (F.length("encrypted_msisdn") > 400)).cast("int")).alias("encrypted_too_long"),
    F.sum((F.col("source_event_id").isNotNull() & (F.length("source_event_id") > 50)).cast("int")).alias("request_id_too_long"),
    F.sum((F.col("response_code").isNotNull() & (F.length("response_code") > 50)).cast("int")).alias("response_code_too_long"),
    F.sum((F.col("source_ip").isNotNull() & (F.length("source_ip") > 255)).cast("int")).alias("source_ip_too_long"),
    F.sum((F.col("brand").isNotNull() & (F.length("brand") > 255)).cast("int")).alias("brand_too_long"),
    F.sum((F.col("operation").isNotNull() & (F.length("operation") > 50)).cast("int")).alias("operation_too_long"),

    # ----- test ID leakage (should be 0) -----
    F.sum(F.col("partner_id").isin(TEST_PARTNER_IDS).cast("int")).alias("leaked_test_partner_ids"),
    F.sum(F.col("service_provider_id").isin(TEST_SERVICE_PROVIDER_IDS).cast("int")).alias("leaked_test_sp_ids"),
).collect()[0].asDict()

_metrics = {"total_rows": _total, **_agg}
print("Computed metrics:")
for k, v in _metrics.items():
    print(f"  {k:40s} = {v:,}" if isinstance(v, int) else f"  {k:40s} = {v}")


In [ ]:
%%pyspark project.spark.compatibility
# ============================================================
# Unit / Invariant tests
# ============================================================
def _df(rows, schema=None):
    return spark.createDataFrame(rows, schema=schema) if schema else spark.createDataFrame(rows)

# ---------- Schema ----------
def test_schema_has_23_columns():
    n = len(df_silver.columns)
    assert n == 23, f"expected 23 cols, got {n}"

def test_schema_column_names_match_spec():
    expected = set(FINAL_COLS); actual = set(df_silver.columns)
    missing = expected - actual; extra = actual - expected
    assert not missing and not extra, f"missing={missing}, extra={extra}"

def test_schema_types():
    expected = {
        "record_id": "string", "operation": "string", "mno": "string", "msisdn": "string",
        "phone_number_AT_hash": "string", "encrypted_msisdn": "string",
        "partner_id": "string", "partner_name": "string",
        "service_provider_id": "string", "service_provider_name": "string",
        "response_code": "string", "source_ip": "string",
        "api_timestamp": "timestamp", "date": "date", "timestamp_ms": "timestamp",
        "brand": "string", "industry": "string", "api_type": "string",
        "processing_time_ms": "bigint",
        "correlation_id": "string", "source_name": "string",
        "source_event_id": "string", "schema_version": "int",
    }
    actual = {f.name: f.dataType.simpleString() for f in df_silver.schema.fields}
    mismatches = [(c, expected[c], actual.get(c)) for c in expected if actual.get(c) != expected[c]]
    assert not mismatches, f"type mismatches: {mismatches}"

# ---------- Null checks (critical columns) ----------
def test_no_null_record_id():            assert _metrics["null_record_id"] == 0
def test_no_null_operation():            assert _metrics["null_operation"] == 0
def test_no_null_partner_id():           assert _metrics["null_partner_id"] == 0
def test_no_null_service_provider_id():  assert _metrics["null_service_provider_id"] == 0
def test_no_null_api_timestamp():        assert _metrics["null_api_timestamp"] == 0
def test_no_null_api_type():             assert _metrics["null_api_type"] == 0
def test_no_null_source_name():          assert _metrics["null_source_name"] == 0
def test_no_null_schema_version():       assert _metrics["null_schema_version"] == 0
def test_no_null_source_event_id():      assert _metrics["null_source_event_id"] == 0

# ---------- Format checks ----------
def test_record_id_is_hex64():               assert _metrics["bad_record_id_format"] == 0
def test_phone_number_AT_hash_is_hex64():   assert _metrics["bad_hash_format"] == 0
def test_encrypted_msisdn_is_base64():      assert _metrics["bad_b64_format"] == 0
def test_source_ip_is_ipv4():               assert _metrics["bad_ipv4_format"] == 0

# ---------- Padding ----------
def test_partner_id_padded_to_10_when_digit():          assert _metrics["partner_id_digit_not_10"] == 0
def test_service_provider_id_padded_to_10_when_digit(): assert _metrics["sp_id_digit_not_10"] == 0

# ---------- Constants ----------
def test_source_name_constant():     assert _metrics["bad_source_name"] == 0
def test_schema_version_constant():  assert _metrics["bad_schema_version"] == 0
def test_correlation_id_all_populated(): assert _metrics["correlation_id_non_null"] == _metrics["total_rows"]
def test_correlation_id_gen_uuid():      assert _metrics["bad_correlation_id_format"] == 0

# ---------- Test ID exclusion (regression) ----------
def test_no_test_partner_ids_leaked():  assert _metrics["leaked_test_partner_ids"] == 0
def test_no_test_sp_ids_leaked():       assert _metrics["leaked_test_sp_ids"] == 0

# ---------- Varchar length constraints ----------
def test_partner_id_within_50_chars():        assert _metrics["partner_id_too_long"]    == 0
def test_sp_id_within_50_chars():             assert _metrics["sp_id_too_long"]         == 0
def test_mno_within_255_chars():              assert _metrics["mno_too_long"]           == 0
def test_msisdn_within_20_chars():            assert _metrics["msisdn_too_long"]        == 0
def test_hashed_msisdn_within_70_chars():     assert _metrics["hashed_msisdn_too_long"] == 0
def test_encrypted_msisdn_within_400_chars(): assert _metrics["encrypted_too_long"]     == 0
def test_request_id_within_50_chars():        assert _metrics["request_id_too_long"]    == 0
def test_response_code_within_50_chars():     assert _metrics["response_code_too_long"] == 0
def test_source_ip_within_255_chars():        assert _metrics["source_ip_too_long"]     == 0
def test_brand_within_255_chars():            assert _metrics["brand_too_long"]         == 0
def test_operation_within_50_chars():         assert _metrics["operation_too_long"]     == 0

# ---------- Dedup invariant (regression for the V2 bug) ----------
def test_no_duplicate_record_ids():
    total    = _metrics["total_rows"]
    distinct = df_silver.select("record_id").distinct().count()
    assert total == distinct, f"{total - distinct} duplicate rows (total={total}, distinct={distinct})"

# ---------- Fixture tests: pad_id_to_10 ----------
def test_pad_id_to_10_short_digit_padded():
    assert _df([Row(x="123")]).withColumn("x", pad_id_to_10("x")).collect()[0]["x"] == "0000000123"
def test_pad_id_to_10_already_10_unchanged():
    assert _df([Row(x="1234567890")]).withColumn("x", pad_id_to_10("x")).collect()[0]["x"] == "1234567890"
def test_pad_id_to_10_mixed_unchanged():
    assert _df([Row(x="abc123")]).withColumn("x", pad_id_to_10("x")).collect()[0]["x"] == "abc123"
def test_pad_id_to_10_alpha_unchanged():
    assert _df([Row(x="testServiceProvider")]).withColumn("x", pad_id_to_10("x")).collect()[0]["x"] == "testServiceProvider"
def test_pad_id_to_10_null_unchanged():
    schema = StructType([StructField("x", StringType(), True)])
    assert _df([Row(x=None)], schema=schema).withColumn("x", pad_id_to_10("x")).collect()[0]["x"] is None

# ---------- Fixture tests: test ID exclusion logic ----------
def test_exclusion_filters_test_partner_id():
    rows = [Row(partner_id="enstream", service_provider_id="real"),
            Row(partner_id="real",     service_provider_id="real")]
    out = _df(rows).filter(
        (~F.col("partner_id").isin(TEST_PARTNER_IDS)) &
        (~F.col("service_provider_id").isin(TEST_SERVICE_PROVIDER_IDS))
    ).collect()
    assert len(out) == 1 and out[0]["partner_id"] == "real"

def test_exclusion_filters_test_sp_id():
    rows = [Row(partner_id="real", service_provider_id="testServiceProvider"),
            Row(partner_id="real", service_provider_id="real")]
    out = _df(rows).filter(
        (~F.col("partner_id").isin(TEST_PARTNER_IDS)) &
        (~F.col("service_provider_id").isin(TEST_SERVICE_PROVIDER_IDS))
    ).collect()
    assert len(out) == 1 and out[0]["service_provider_id"] == "real"

print("All test functions defined.")


In [ ]:
%%pyspark project.spark.compatibility
# Run all unit tests
_tests = [
    # Schema
    test_schema_has_23_columns, test_schema_column_names_match_spec, test_schema_types,
    # Null checks
    test_no_null_record_id, test_no_null_operation, test_no_null_partner_id,
    test_no_null_service_provider_id, test_no_null_api_timestamp, test_no_null_api_type,
    test_no_null_source_name, test_no_null_schema_version, test_no_null_source_event_id,
    # Format
    test_record_id_is_hex64, test_phone_number_AT_hash_is_hex64,
    test_encrypted_msisdn_is_base64, test_source_ip_is_ipv4,
    # Padding
    test_partner_id_padded_to_10_when_digit, test_service_provider_id_padded_to_10_when_digit,
    # Constants
    test_source_name_constant, test_schema_version_constant, test_correlation_id_all_populated, test_correlation_id_gen_uuid,
    # Test ID exclusion
    test_no_test_partner_ids_leaked, test_no_test_sp_ids_leaked,
    # Varchar length
    test_partner_id_within_50_chars, test_sp_id_within_50_chars, test_mno_within_255_chars,
    test_msisdn_within_20_chars, test_hashed_msisdn_within_70_chars,
    test_encrypted_msisdn_within_400_chars, test_request_id_within_50_chars,
    test_response_code_within_50_chars, test_source_ip_within_255_chars,
    test_brand_within_255_chars, test_operation_within_50_chars,
    # Dedup invariant
    test_no_duplicate_record_ids,
    # Fixture: pad_id_to_10
    test_pad_id_to_10_short_digit_padded, test_pad_id_to_10_already_10_unchanged,
    test_pad_id_to_10_mixed_unchanged, test_pad_id_to_10_alpha_unchanged,
    test_pad_id_to_10_null_unchanged,
    # Fixture: exclusion logic
    test_exclusion_filters_test_partner_id, test_exclusion_filters_test_sp_id,
]
_failed = 0
for _t in _tests:
    try:
        _t()
        print(f"PASS  {_t.__name__}")
    except AssertionError as e:
        _failed += 1
        print(f"FAIL  {_t.__name__}: {e}")
    except Exception as e:
        _failed += 1
        print(f"ERROR {_t.__name__}: {type(e).__name__}: {e}")
print()
print(f"{len(_tests) - _failed}/{len(_tests)} passed")
